## Practice Notebook: Systems of Linear Equations — Direct Methods

---

This notebook develops the **direct methods** for solving $A\mathbf{x}=\mathbf{b}$.

Each section contains essential concepts, pseudocode, Python code, and worked examples.

Each method *improves* on the previous one. The sequence is a ladder of gains in **efficiency** and **numerical stability**, not a list of unrelated recipes.

**Outline:**

1. Method of Elimination (**Gaussian Elimination**)
2. Inverse and Factorization (**LU Decomposition**)
3. Direct Method of Factorization (**Doolittle Algorithm**)
4. Pivoting Strategies:
    - 4a. **Partial Pivoting ($PA = LU$)**
    - 4b. Scaled Partial Pivoting (self-study)
5. Special Matrices and Special Systems (**Cholesky Decomposition**)
  
---

### References

**Notes and examples are inspired by the discussion in the following references**

*Some of the content is freely available online:*

* [Gilbert Strang, *Introduction to Linear Algebra*](https://math.mit.edu/~gs/linearalgebra/)

  
* [N. S. Asaithambi, *Numerical Analysis: Theory and Practice*](https://archive.org/details/numericalanalysi0000asai)


In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)   # readable arrays

## <span style="color:#455a64">Linear Systems of Equations</span>

A [linear system of equations](https://en.wikipedia.org/wiki/System_of_linear_equations) is a collection of linear equations in the same unknowns:

\begin{align}
a_{0,0}x_0 + a_{0,1}x_1 + \cdots + a_{0,n}x_n & = b_0 \\
a_{1,0}x_0 + a_{1,1}x_1 + \cdots + a_{1,n}x_n & = b_1 \\
& \vdots \\
a_{m,0}x_0 + a_{m,1}x_1 + \cdots + a_{m,n}x_n & = b_m
\end{align}

In matrix notation this is $A\mathbf{x}=\mathbf{b}$ with

$$
A = \begin{bmatrix}
a_{0,0} & a_{0,1} & \cdots & a_{0,n} \\
a_{1,0} & a_{1,1} & \cdots & a_{1,n} \\
\vdots & & & \vdots \\
a_{m,0} & a_{m,1} & \cdots & a_{m,n}
\end{bmatrix},\qquad
\mathbf{x} = \begin{bmatrix} x_0 \\ x_1 \\ \vdots \\ x_n \end{bmatrix},\qquad
\mathbf{b} = \begin{bmatrix} b_0 \\ b_1 \\ \vdots \\ b_m \end{bmatrix}.
$$

* $A$ is the **coefficient matrix**, $\mathbf{x}$ the **unknown vector**, $\mathbf{b}$ the **right-hand-side vector**.

**Column picture.** $\mathbf{b}$ is a **linear combination of the columns** of $A$ with weights $x_0,\dots,x_n$:

$$\mathbf{b} = x_0\mathbf{a}_0 + x_1\mathbf{a}_1 + \cdots + x_n\mathbf{a}_n.$$

Solving $A\mathbf{x}=\mathbf{b}$ means finding the weights so that this combination of columns equals $\mathbf{b}$.

For a square system of linear equations, a **unique** solution exists precisely when $A$ has **linearly independent columns** (full rank).


## <span style="color:#455a64">Direct vs. Iterative Methods</span>

**Direct methods**: reach the exact solution (in exact arithmetic) in a **finite, predictable number of operations**

**Iterative methods**: start from a guess solution and refine it recursively

## <span style="color:#1f4e79">1. Gaussian Elimination</span>

We focus on a square system of linear equations.

[Gaussian elimination](https://en.wikipedia.org/wiki/Gaussian_elimination) solves $A\mathbf{x}=\mathbf{b}$ in two phases:

1. **Forward elimination** — reduce the system to an equivalent **upper-triangular** system using elementary row operations.
2. **Back substitution** — solve the triangular system from the bottom equation upward.

A **unique** solution exists when $A$ has **linearly independent columns** (full rank).

### The forward elimination

Turn matrix $A$ into an **upper-triangular** matrix $U$ (row-echelon form).

$$
E A =
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & -\ell_{32} & 1 \end{bmatrix}}_{E_{32}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ -\ell_{31} & 0 & 1 \end{bmatrix}}_{E_{31}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ -\ell_{21} & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}}_{E_{21}}
\underbrace{\begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \end{bmatrix}}_{A}
=
\underbrace{\begin{bmatrix} u_{11} & u_{12} & u_{13} \\ 0 & u_{22} & u_{23} \\ 0 & 0 & u_{33} \end{bmatrix}}_{U}
$$

where, multipliers are given as:

* $\ell_{21} = a_{21}/a_{11}$
* $\ell_{31} = a_{31}/a_{11}$
* $\ell_{32} = a_{32}^*/a_{22}^* = \dfrac{a_{32} - \ell_{31}\,a_{12}}{a_{22} - \ell_{21}\,a_{12}}$

### The back substitution

$$
\underbrace{\begin{bmatrix} u_{11} & u_{12} & u_{13} \\ 0 & u_{22} & u_{23} \\ 0 & 0 & u_{33} \end{bmatrix}}_{U}
\underbrace{\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix} c_1 \\ c_2 \\ c_3 \end{bmatrix}}_{\mathbf{c}}
$$

We solve **from the bottom up**:

Solution in general form:

$$
x_i = \frac{1}{u_{ii}}\left( c_i - \sum_{j=i+1}^{n} u_{ij}\,x_j \right),
\qquad i = n, n-1, \ldots, 1
$$

### Pseudocode — Gaussian elimination

```text
INPUT : n x n matrix a, n-vector b
OUTPUT: solution vector x


# ---- Forward elimination ----
for k = 1 to n-1:                       # pivot columns
    if a(k,k) == 0:                     # need a nonzero pivot
        find p in {k+1..n} with a(p,k) != 0
        if none: stop "no unique solution"
        swap row k <-> row p ; swap b(k) <-> b(p)

    for i = k+1 to n:                   # rows below the pivot
        l = a(i,k) / a(k,k)             # multiplier (l)
        for j = k to n:
            a(i,j) = a(i,j) - l * a(k,j)
        b(i) = b(i) - l * b(k)

# ---- Back substitution ----
for i = n down to 1:
    s = b(i) - sum_{j=i+1..n} a(i,j) * x(j)
    x(i) = s / a(i,i)
```


### Python — Back Substitution

In [ ]:
def back_substitution(U, b):
    "Solve U x = b for upper-triangular U (nonzero diagonal)."
    n = len(b)
    x = np.zeros(n)

    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

### Turning a summation into a vector product

The back-substitution formula contains a sum:

$$\sum_{j=i+1}^{n} u_{ij}\,x_j = u_{i,i+1}x_{i+1} + u_{i,i+2}x_{i+2} + \cdots + u_{i,n}x_n$$

The **dot product** of two vectors with the `@` operator:

$$
\underbrace{\begin{bmatrix} u_{i,i+1} & u_{i,i+2} & \cdots & u_{i,n} \end{bmatrix}}_{\texttt{U[i, i+1:]}}
\cdot
\underbrace{\begin{bmatrix} x_{i+1} \\ x_{i+2} \\ \vdots \\ x_n \end{bmatrix}}_{\texttt{x[i+1:]}}
\;=\; \sum_{j=i+1}^{n} u_{ij}\,x_j
$$

**What the two slices grab**

* `U[i, i+1:]` — row `i` of `U`, keeping only the columns *to the right* of the diagonal.
* `x[i+1:]` — the unknowns *below* `x[i]`, which have already been computed.

### Python — Gaussian Elimination

In [ ]:
def gauss_elimination(A, b):
    "Solve Ax=b by Gaussian elimination; swaps only if a pivot is exactly zero."
    A = A.astype(float).copy()
    b = b.astype(float).copy()
    n = len(b)

    for k in range(n - 1):
        if A[k, k] == 0:                                   # zero pivot -> swap
            swap = np.argmax(np.abs(A[k+1:, k])) + (k + 1)
            if A[swap, k] == 0:
                raise ValueError("No unique solution (singular matrix).")
            A[[k, swap]] = A[[swap, k]]
            b[[k, swap]] = b[[swap, k]]

        l = A[k+1:, k] / A[k, k]                           # multipliers (vector)
        A[k+1:, k:] -= np.outer(l, A[k, k:])               # row update
        b[k+1:]     -= l * b[k]

    return back_substitution(A, b)

```text
 ---- Forward elimination (pseudocode) ----

for k = 1 to n-1:                       # pivot columns
    if a(k,k) == 0:                     # need a nonzero pivot
        find p in {k+1..n} with a(p,k) != 0
        if none: stop "no unique solution"
        swap row k <-> row p ; swap b(k) <-> b(p)

    for i = k+1 to n:                   # rows below the pivot
        l = a(i,k) / a(k,k)             # multiplier (l)
        for j = k to n:
            a(i,j) = a(i,j) - l * a(k,j)
        b(i) = b(i) - l * b(k)
```
The for loop part that calculates the multiplier and updates the row below the pivot:

$$\ell_i = \frac{a_{ik}}{a_{kk}}, \qquad \text{Row}_i \leftarrow \text{Row}_i - \ell_i \cdot \text{Row}_k$$

Replacing it with vectorized version:

```text
l = A[k+1:, k] / A[k, k]                           # multipliers (vector)
A[k+1:, k:] -= np.outer(l, A[k, k:])               # row update
b[k+1:]     -= l * b[k]


---  Understanding the vectorized part ---
```

$$A=\begin{bmatrix}2&3&4\\4&11&14\\2&8&17\end{bmatrix}$$

* `k = 0`
* pivot row `A[0, 0:]` $=[2,\ 3,\ 4]$
* multipliers `l` = `A[1:,0]/A[0,0]` $= [4/2,\ 2/2] = [2,\ 1]$

$$
\texttt{np.outer(l, pivot)} =
\begin{bmatrix} 2 \\ 1 \end{bmatrix}\begin{bmatrix} 2 & 3 & 4 \end{bmatrix}
= \begin{bmatrix} 4 & 6 & 8 \\ 2 & 3 & 4 \end{bmatrix}
$$

* Subtracting `A[k+1:, k:] - np.outer(l, pivot)`

$$\begin{bmatrix}4&11&14\\2&8&17\end{bmatrix} -  \begin{bmatrix} 4 & 6 & 8 \\ 2 & 3 & 4 \end{bmatrix} = \begin{bmatrix}0&5&6\\0&5&13\end{bmatrix}$$

**The right-hand side**

$$b = \begin{bmatrix} 19 \\ 55 \\ 50\end{bmatrix}$$

* `l * b[k]`

$$\begin{bmatrix} 2 \\ 1 \end{bmatrix}\begin{bmatrix} 19 \end{bmatrix}
= \begin{bmatrix} 38 \\ 19 \end{bmatrix}$$

* `b[k+1:] - l * b[k]`

$$\begin{bmatrix} 55 \\ 50 \end{bmatrix} - \begin{bmatrix} 38 \\ 19 \end{bmatrix}
= \begin{bmatrix} 17 \\ 31 \end{bmatrix}$$

### Example

$$
\begin{bmatrix} 2 & 3 & 4 \\ 4 & 11 & 14 \\ 2 & 8 & 17 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
= \begin{bmatrix} 19 \\ 55 \\ 50 \end{bmatrix}
$$

In [ ]:
A = np.array([[2, 3, 4],
              [4, 11, 14],
              [2, 8, 17]], dtype=float)
b = np.array([19, 55, 50], dtype=float)

x = gauss_elimination(A, b)
print("x =", x)

x = [4. 1. 2.]


### Example

$$
\begin{bmatrix} 2 & 3 & 4 \\ 4 & 6 & 14 \\ 2 & 8 & 17 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
= \begin{bmatrix} 19 \\ 55 \\ 50 \end{bmatrix}
$$

In [ ]:
A = np.array([[2, 3, 4],
              [4, 6, 14],
              [2, 8, 17]], dtype=float)
b = np.array([19, 55, 50], dtype=float)

x = gauss_elimination(A, b)
print("x =", x)

print("NumPy reference =", np.linalg.solve(A, b))

x = [ 5.583333 -1.166667  2.833333]
NumPy reference = [ 5.583333 -1.166667  2.833333]


### Example

$$\begin{aligned}\phantom{2x_1 + {}}x_2 + x_3 &= 4\\ x_1 + 2x_2 + x_3 &= 6\\ 2x_1 + x_2 + x_3 &= 6\end{aligned}$$

In [ ]:
A = np.array([[0, 1, 1],
              [1, 2, 1],
              [2, 1, 1]], dtype=float)
b = np.array([4, 6, 6], dtype=float)

x = gauss_elimination(A, b)
print("x =", x)

print("NumPy reference =", np.linalg.solve(A, b))

x = [1. 1. 3.]
NumPy reference = [1. 1. 3.]


In [ ]:
import numpy as np

A = np.array([[2, 1], [2, 1.02]])
print(A)

b = np.array([3, 0])

x = gauss_elimination(A, b)
print("x =", x)

[[2.   1.  ]
 [2.   1.02]]
x = [  76.5 -150. ]


### Arithmetic cost

* Additions/subtractions are cheaper than multiplications/divisions, but both scale the same way here.
* Forward elimination dominates: $\approx \tfrac{2n^3}{3}$ operations. Back substitution is only $\approx n^2$.


## <span style="color:#2e7d32">2. Inverse and Factorization — LU Decomposition</span>

*Gaussian elimination repeats the full elimination for every new $\mathbf{b}$. Factorization does that work once, then reuses it — our first gain in efficiency.*

The idea of factorization (or decomposition) builds on the forward elimination:

$$
E A =
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & -\ell_{32} & 1 \end{bmatrix}}_{E_{32}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ -\ell_{31} & 0 & 1 \end{bmatrix}}_{E_{31}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ -\ell_{21} & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}}_{E_{21}}
\underbrace{\begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \end{bmatrix}}_{A} =
\underbrace{\begin{bmatrix} u_{11} & u_{12} & u_{13} \\ 0 & u_{22} & u_{23} \\ 0 & 0 & u_{33} \end{bmatrix}}_{U}
$$

Taking the inverse of the elimination matrices:

$$
\underbrace{\begin{bmatrix} a_{11} & a_{12} & a_{13} \\ a_{21} & a_{22} & a_{23} \\ a_{31} & a_{32} & a_{33} \end{bmatrix}}_{A} =
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ \ell_{21} & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}}_{E_{21}^{-1}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ \ell_{31} & 0 & 1 \end{bmatrix}}_{E_{31}^{-1}}
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 0 & 1 & 0 \\ 0 & \ell_{32} & 1 \end{bmatrix}}_{E_{32}^{-1}}
\underbrace{\begin{bmatrix} u_{11} & u_{12} & u_{13} \\ 0 & u_{22} & u_{23} \\ 0 & 0 & u_{33} \end{bmatrix}}_{U}
$$

$$
A =
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ \ell_{21} & 1 & 0 \\ \ell_{31} & \ell_{32} & 1 \end{bmatrix}}_{L}
\underbrace{\begin{bmatrix} u_{11} & u_{12} & u_{13} \\ 0 & u_{22} & u_{23} \\ 0 & 0 & u_{33} \end{bmatrix}}_{U}
$$

**Why is having the factorization $A=LU$ helpful?**

*Adds efficiency:* Reuse the factors to determine $\mathbf{x}$ for *any* new $\mathbf{b}$ later.

* $A\mathbf{x}=\mathbf{b}$
* $LU\mathbf{x}=\mathbf{b}$
* $L\mathbf{c}=\mathbf{b}$ -- find $\mathbf{c}$ by forward substitution, requires $n^2$ operations
* $U\mathbf{x}=\mathbf{c}$ -- find $\mathbf{x}$ by back substitution, requires $n^2$ operations

Factor **once** at $\mathcal{O}(n^3)$; then every additional load vector costs only $\mathcal{O}(n^2)$.

**"What else does LU give us?"**

The determinant of $A$ is a by-product of the factorization, very easily and efficiently computed. Since $A = LU$ with $L$ unit lower triangular ($\det L = 1$) and $U$ upper triangular, the determinant is simply the *product of the pivots*:

$$\det(A) = \det(L)\,\det(U) = 1 \times \big(u_{11}\,u_{22}\cdots u_{nn}\big).$$

The determinant also tells us the **nature of the solution**:

* $\det(A) \neq 0$ (all pivots nonzero) $\Rightarrow$ $A$ is nonsingular / invertible $\Rightarrow$ a **unique** solution exists.
* $\det(A) = 0$ (a zero pivot appears) $\Rightarrow$ $A$ is singular $\Rightarrow$ **no unique** solution (either none or infinitely many).

In [ ]:
def forward_substitution(L, b):
    "Solve L y = b for lower-triangular L (nonzero diagonal)."
    n = L.shape[0]
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

def back_substitution(U, y):
    "Solve U x = y for upper-triangular U (nonzero diagonal)."
    n = U.shape[0]
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

# Based on Gaussian Elimination (Outer-Product / Submatrix update)
def lu_factor_nopivot(A):
    "Factor A = L U (unit lower L, upper U). Assumes no zero pivots."
    A = A.astype(float).copy(); n = A.shape[0]
    L = np.eye(n); U = A.copy()
    for k in range(n - 1):
        if U[k, k] == 0:
            raise ValueError("Zero pivot encountered")
        f = U[k+1:, k] / U[k, k]        # multipliers for the sub-column
        L[k+1:, k] = f                  # store them in L
        U[k+1:, k:] -= np.outer(f, U[k, k:])
    return L, U

def lu_solve(L, U, b):
    "Solve A x = b given A = L U, via forward then back substitution."
    y = forward_substitution(L, b)      # L y = b
    x = back_substitution(U, y)      # U x = y
    return x

### Example

$$
\begin{bmatrix} 2 & 3 & 4 \\ 4 & 11 & 14 \\ 2 & 8 & 17 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
= \begin{bmatrix} 19 \\ 55 \\ 50 \end{bmatrix}
$$

In [ ]:
A = np.array([
    [2.0,  3.0,  4.0],
    [4.0, 11.0, 14.0],
    [2.0,  8.0, 17.0]
], dtype=float)

b = np.array([19.0, 55.0, 50.0], dtype=float)

# 1. Compute LU Factorization (A = L * U)
L, U = lu_factor_nopivot(A)

# 2. Solve Ax = b via Forward and Backward Substitution
x = lu_solve(L, U, b)

# Display Results
print("--- Matrix L (Unit Lower Triangular) ---")
print(L)

print("\n--- Matrix U (Upper Triangular) ---")
print(U)

print("\n--- Verification: Reconstructed A (L @ U) ---")
print(L @ U)

print("\n--- Final Solution Vector x ---")
print(x)

--- Matrix L (Unit Lower Triangular) ---
[[1. 0. 0.]
 [2. 1. 0.]
 [1. 1. 1.]]

--- Matrix U (Upper Triangular) ---
[[2. 3. 4.]
 [0. 5. 6.]
 [0. 0. 7.]]

--- Verification: Reconstructed A (L @ U) ---
[[ 2.  3.  4.]
 [ 4. 11. 14.]
 [ 2.  8. 17.]]

--- Final Solution Vector x ---
[4. 1. 2.]


In [ ]:
### Example

In [ ]:
A = np.array([[2, 3, 4],
              [4, 11, 14],
              [2, 8, 17]], dtype=float)

L, U = lu_factor_nopivot(A)          # factorization from earlier in the notebook

det_from_pivots = np.prod(np.diag(U))     # product of the pivots
print("pivots (diagonal of U):", np.diag(U))
print("det from pivots        :", det_from_pivots)
print("np.linalg.det (check)  :", np.linalg.det(A))

pivots (diagonal of U): [2. 5. 7.]
det from pivots        : 70.0
np.linalg.det (check)  : 70.00000000000003


### Elimination Matrices and Inverse Matrices

Each elementary row operation is itself a matrix.

$$
E_{21} A =
\begin{bmatrix} 1 & 0 & 0 \\ -2 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}
\begin{bmatrix} 2 & 3 & 4 \\ 4 & 11 & 14 \\ 2 & 8 & 17 \end{bmatrix}
= \begin{bmatrix} 2 & 3 & 4 \\ 0 & 5 & 6 \\ 2 & 8 & 17 \end{bmatrix}
$$

Elimination $E_{21}$ subtracts; the inverse $E_{21}^{-1}$ does its opposite (adds):

$$
A =
\begin{bmatrix} 2 & 3 & 4 \\ 4 & 11 & 14 \\ 2 & 8 & 17 \end{bmatrix} =
\underbrace{\begin{bmatrix} 1 & 0 & 0 \\ 2 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}}_{E_{21}^{-1}}
\begin{bmatrix} 2 & 3 & 4 \\ 0 & 5 & 6 \\ 2 & 8 & 17 \end{bmatrix}
$$

* $A$ is a square matrix and its **inverse matrix** $A^{-1}$ is of the same size, so that $A^{-1}$ times $A$ equals $I$.
  
* Whatever $A$ does, $A^{-1}$ undoes.

* If $A$ and $B$ are invertible (same size), then the inverse of $AB$ is $B^{-1}A^{-1}$.

* $A^{-1}$ exists if and only if $A$ is **nonsingular** — equivalently, $\det(A) \neq 0$, all $n$ pivots are nonzero, and the columns of $A$ are linearly independent.

**How to Compute the Inverse of a Square Matrix $A$?**

We are solving $n$ equations that all share the same coefficient matrix $A$.

**Gauss–Jordan elimination:** Instead of a matrix $[\,A \mid \mathbf{b}\,]$ augmented by one right-hand side $\mathbf{b}$, we use a matrix $[\,A \mid I\,]$ augmented by $n$ right-hand sides (the columns of $I$). Then, as $A$ changes to $I$, elimination produces $[\,I \mid A^{-1}\,]$.

$$
[\,A \mid I\,] \;\longrightarrow\; [\,I \mid A^{-1}\,]
$$

*Note:* Gauss–Jordan goes further than Gaussian elimination — it eliminates entries **both below and above** each pivot and scales every pivot to $1$, reducing $A$ all the way to $I$ (not just to an upper-triangular $U$).

---

**Why we usually do not compute $A^{-1}$**

Forming $A^{-1}$ effectively solves $AX = I$ — $n$ right-hand sides, about $n^3$ work. Solving the single system by one elimination is far cheaper. We build $A^{-1}$ here only to *understand* inverses and Gauss–Jordan; for real solving we use elimination / $LU$. Gauss–Jordan itself costs $\approx \tfrac{n^3}{2}$ versus $\tfrac{n^3}{3}$ for elimination + back-substitution, so it is mainly a conceptual and inverse-building tool.

---

## <span style="color:#6a1b9a">3. Direct Factorization — Doolittle Algorithm</span>

*Doolittle builds the same $L$ and $U$ directly by matching entries of $A = LU$. This is the compact, in-place form linear-algebra libraries actually use.*

Direct $LU$ factorization is the approach used in linear-algebra libraries.

The core idea of direct $LU$ factorization is to decompose a non-singular $n \times n$ matrix $A$ into a product of a lower triangular matrix $L$ and an upper triangular matrix $U$:

$$A = LU$$

$$\begin{bmatrix}  a_{11} & a_{12} & \dots & a_{1n} \\  a_{21} & a_{22} & \dots & a_{2n} \\  \vdots & \vdots & \ddots & \vdots \\  a_{n1} & a_{n2} & \dots & a_{nn}  \end{bmatrix}  =  \begin{bmatrix}  l_{11} & 0 & \dots & 0 \\  l_{21} & l_{22} & \dots & 0 \\  \vdots & \vdots & \ddots & \vdots \\  l_{n1} & l_{n2} & \dots & l_{nn}  \end{bmatrix} \begin{bmatrix}  u_{11} & u_{12} & \dots & u_{1n} \\  0 & u_{22} & \dots & u_{2n} \\  \vdots & \vdots & \ddots & \vdots \\  0 & 0 & \dots & u_{nn}  \end{bmatrix}$$


**Is the factorization $A = LU$ unique?**

* Total unknowns in $L$ and $U$ = $n^2 + n$
* For each entry $a_{ij}$ of $A$ we get one equation — $n^2$ equations in total
* Degrees of Freedom = Total Unknowns - Total Equations = $(n^2 + n) - n^2 = n$

Without additional constraints, the standard factorization $A = LU$ has infinitely many solutions. To obtain a unique solution, we must arbitrarily fix $n$ unknown values in our system.

* **Doolittle Algorithm:** Fixes the diagonal of $L$ to ones ($l_{ii} = 1$ for all $i$)
* **Crout Algorithm:** Fixes the diagonal of $U$ to ones ($u_{ii} = 1$ for all $i$)

For example, say we want to factor $A = LU$ for a $3 \times 3$ matrix:

$$\begin{bmatrix}  a_{11} & a_{12} & a_{13} \\  a_{21} & a_{22} & a_{23} \\  a_{31} & a_{32} & a_{33}  \end{bmatrix}  =  \begin{bmatrix}  l_{11} & 0 & 0 \\  l_{21} & l_{22} & 0 \\  l_{31} & l_{32} & l_{33}  \end{bmatrix}  \begin{bmatrix}  u_{11} & u_{12} & u_{13} \\  0 & u_{22} & u_{23} \\  0 & 0 & u_{33}  \end{bmatrix}$$

By definition of matrix multiplication, every element $a_{ij}$ is the dot product of row $i$ of $L$ and column $j$ of $U$:

$$a_{ij} = \sum_{k=1}^{n} l_{ik} u_{kj}$$

Since $L$ is lower triangular $(l_{ik} = 0$ for $k > i)$ and $U$ is upper triangular $(u_{kj} = 0$ for $k > j)$, this summation stops at $k = \min(i, j)$:

$$a_{ij} = \sum_{k=1}^{\min(i,j)} l_{ik} u_{kj}$$

Carrying out the row-by-column matrix multiplication yields 9 scalar equations:

$$\begin{bmatrix}  a_{11} & a_{12} & a_{13} \\  a_{21} & a_{22} & a_{23} \\  a_{31} & a_{32} & a_{33}  \end{bmatrix}  =  \begin{bmatrix}  l_{11} u_{11} & l_{11} u_{12} & l_{11} u_{13} \\  l_{21} u_{11} & l_{21} u_{12} + l_{22} u_{22} & l_{21} u_{13} + l_{22} u_{23} \\  l_{31} u_{11} & l_{31} u_{12} + l_{32} u_{22} & l_{31} u_{13} + l_{32} u_{23} + l_{33} u_{33}  \end{bmatrix}$$



**Step 1:** Set Diagonal of $L = 1$

* $$l_{11} = 1, \quad l_{22} = 1, \quad l_{33} = 1$$

$$\begin{bmatrix}  a_{11} & a_{12} & a_{13} \\  a_{21} & a_{22} & a_{23} \\  a_{31} & a_{32} & a_{33}  \end{bmatrix}  =  \begin{bmatrix}  u_{11} & u_{12} & u_{13} \\  l_{21} u_{11} & l_{21} u_{12} + u_{22} & l_{21} u_{13} + u_{23} \\  l_{31} u_{11} & l_{31} u_{12} + l_{32} u_{22} & l_{31} u_{13} + l_{32} u_{23} + u_{33}  \end{bmatrix}$$

The equations naturally decouple if we compute them in a specific sequence, i.e. by alternating between calculating a row of $U$ and a column of $L$

**Step 2:** Compute Row 1 of $U$

* $u_{11} = a_{11}$
* $u_{12} = a_{12}$
* $u_{13} = a_{13}$

**Step 3:** Compute Column 1 of $L$

* $l_{21} = \frac{a_{21}}{u_{11}}$
* $l_{31} = \frac{a_{31}}{u_{11}}$

**Step 4:** Compute Row 2 of $U$
* $u_{22} = a_{22} - l_{21} u_{12}$
* $u_{23} = a_{23} - l_{21} u_{13}$

**Step 5:** Compute Column 2 of $L$

* $l_{32} = \frac{a_{32} - l_{31} u_{12}}{u_{22}}$

**Step 6:** Compute Row 3 of $U$

* $u_{33} = a_{33} - l_{31} u_{13} - l_{32} u_{23}$

---

General $n \times n$ Doolittle Formulation:

* Diagonal of $L$:

$$l_{ii} = 1 \quad \text{for } i = 1, 2, \dots, n$$

* Row $i$ of $U$ (for $j = i, i+1, \dots, n$):

$u_{ij}$ $\rightarrow$ Row $i$ fixed, Column $j$ moving

$$u_{ij} = a_{ij} - \sum_{k=1}^{i-1} l_{ik} u_{kj}$$

* Column $i$ of $L$ (for $j = i+1, i+2, \dots, n$):

$l_{ji}$ $\rightarrow$ Row $j$ moving, Column $i$ fixed

$$l_{ji} = \frac{1}{u_{ii}} \left( a_{ji} - \sum_{k=1}^{i-1} l_{jk} u_{ki} \right)$$


---

### Algorithm: Doolittle_LU_Decomposition(A)

```text
Input:  A -> (n x n) non-singular matrix
Output: L -> (n x n) lower triangular matrix with 1s on diagonal
        U -> (n x n) upper triangular matrix

Initialize L as an (n x n) identity matrix (I)
Initialize U as an (n x n) zero matrix

For i = 1 to n:
    
    # 1. Compute Row i of U
    For j = i to n:
        sum_u = 0.0
        For k = 1 to i - 1:
            sum_u = sum_u + (L[i, k] * U[k, j])
        U[i, j] = A[i, j] - sum_u
        
    # Check for zero pivot to prevent division by zero
    If U[i, i] == 0:
        Raise Error("Zero pivot encountered! Doolittle LU without pivoting fails.")
        
    # 2. Compute Column i of L (for entries below diagonal)
    For j = i + 1 to n:
        sum_l = 0.0
        For k = 1 to i - 1:
            sum_l = sum_l + (L[j, k] * U[k, i])
        L[j, i] = (A[j, i] - sum_l) / U[i, i]

Return L, U
```

In [ ]:
## Doolittle's Algorithm (Inner-Product / Dot-Product)
def doolittle_lu(A):
    A = A.astype(np.float64).copy(); n = A.shape[0]
    L = np.eye(n, dtype=np.float64); U = np.zeros((n, n),dtype=np.float64)

    for i in range(n):

        for j in range(i, n):
            U[i, j] = A[i, j] - np.dot(L[i, :i], U[:i, j])

        if U[i, i] == 0:
            raise ValueError("Zero pivot encountered")

        for j in range(i + 1, n):
            L[j, i] = (A[j, i] - np.dot(L[j, :i], U[:i, i])) / U[i, i]

    return L, U

### Example

$$
\begin{bmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
= \begin{bmatrix} 4 \\ 8 \\ 20 \end{bmatrix}
$$

In [ ]:
import numpy as np

In [ ]:
# Matrix & RHS
A1 = np.array([
    [2.0, 1.0, 1.0],
    [4.0, 3.0, 3.0],
    [8.0, 7.0, 9.0]
])
b1 = np.array([4.0, 8.0, 20.0])

# Compute factorizations
L_ge, U_ge = lu_factor_nopivot(A1)
L_doo, U_doo = doolittle_lu(A1)

# Compare L and U
print("--- Example: Matrix Factorization Comparison ---\n")
print("L (Gaussian Elimination):\n", L_ge)
print("L (Doolittle):\n", L_doo)
print("\nU (Gaussian Elimination):\n", U_ge)
print("U (Doolittle):\n", U_doo)

# Verify differences
print("\nDifference in L:", np.max(np.abs(L_ge - L_doo)))
print("Difference in U:", np.max(np.abs(U_ge - U_doo)))

# Solve system
x1 = lu_solve(L_doo, U_doo, b1)
print("\nSolution x:", x1)

--- Example: Matrix Factorization Comparison ---

L (Gaussian Elimination):
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]]
L (Doolittle):
 [[1. 0. 0.]
 [2. 1. 0.]
 [4. 3. 1.]]

U (Gaussian Elimination):
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]
U (Doolittle):
 [[2. 1. 1.]
 [0. 1. 1.]
 [0. 0. 2.]]

Difference in L: 0.0
Difference in U: 0.0

Solution x: [ 2. -2.  2.]


**Why Doolittle over standard Gaussian elimination for finding LU?**

Doolittle computes each entry as a single inner product vs. an outer product in Gaussian.

Standard Gaussian elimination repeatedly updates the entire remaining matrix step-by-step, which forces slow memory reads and build up intermediate rounding errors. In contrast, Doolittle calculates each final element directly using a single inner product, allowing to solve efficiently and reduce rounding errors.

## <span style="color:#c62828">4. Pivoting Strategies</span>

### 4a. Partial Pivoting ($PA = LU$)

*So far we assumed every pivot is safe. A zero or tiny pivot corrupts the answer — pivoting restores numerical stability.*

So far we assumed Gaussian elimination and $LU$ factorization proceed smoothly. Two problems break that assumption:

* If a diagonal entry $u_{kk} = 0$ appears, the algorithm halts on division by zero. A row exchange fixes it. *(we already took care of this!)*
  
* If a pivot $u_{kk}$ is extremely small (but nonzero), the algorithm runs without error, yet roundoff can completely corrupt the solution.

Partial pivoting fixes both:

$$PA = LU$$

where $P$ is a permutation matrix that records the row swaps.

## Example

Consider the system $Ax = b$ with very small matrix element

$$\begin{bmatrix} 0.003 & 59.14 \\ 5.291 & -6.130 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} 59.17 \\ 46.78 \end{bmatrix}$$

In [ ]:
A = np.array([
    [0.003, 59.14],
    [5.291, -6.130]
],dtype = np.float16)

b = np.array([59.17, 46.78], dtype=np.float16)

In [ ]:
L, U = doolittle_lu(A)
x_computed = lu_solve(L, U, b)

In [ ]:
print(x_computed)

[19717.045137     0.      ]


In [ ]:
A = np.array([
    [5.291, -6.130],
    [0.003, 59.14]
],dtype=np.float16)

b = np.array([46.78, 59.17], dtype=np.float16)

In [ ]:
L, U = doolittle_lu(A)
x_computed = lu_solve(L, U, b)

In [ ]:
print(x_computed)

[10.003785  1.00008 ]


## Example

Consider the system $Ax = b$ with very small matrix element

$$\begin{bmatrix} 10^{-16} & 1.00 \\ 1.00 & 1.00 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} 1.00 \\ 2.00 \end{bmatrix}$$

In [ ]:
eps = 1e-16

l_12 = 1/eps
print(l_12)

1e+16


In [ ]:
u22 = 1 - l_12
print(u22)

-1e+16


In [ ]:
b2 = 2 - l_12
print(b2)

-9999999999999998.0


In [ ]:
x2 = b2/u22
print(x2)

0.9999999999999998


In [ ]:
x1 = (1 - x2)/eps
print(x1)

2.220446049250313


In [ ]:
# 1. Define epsilon near float64 machine precision (1e-16)
eps = 1e-16

A = np.array([
    [eps, 1.0],
    [1.0, 1.0]
], dtype=np.float64)

b = np.array([1.0, 2.0], dtype=np.float64)

# 2. Solve using unpivoted Doolittle LU
L, U = doolittle_lu(A)
x_computed = lu_solve(L, U, b)

# 3. True exact solution: x1 = 1/(1-eps) ≈ 1.0, x2 = (1-2eps)/(1-eps) ≈ 1.0
x_exact = np.array([1.0 / (1.0 - eps), (1.0 - 2.0 * eps) / (1.0 - eps)])

# --- Display Results ---
#print("--- Matrix L ---")
#print(L)

#print("\n--- Matrix U ---")
#print(U)

print("\n--- Numerical Solution Comparison ---")
print("Doolittle (No Pivoting) x :", x_computed)
print("Exact Mathematical Solution x :", x_exact)
print("Relative Error in x1         :", abs(x_computed[0] - x_exact[0]) / x_exact[0] * 100, "%")


--- Numerical Solution Comparison ---
Doolittle (No Pivoting) x : [2.220446 1.      ]
Exact Mathematical Solution x : [1. 1.]
Relative Error in x1         : 122.04460492503127 %


### Why Small Pivots are Dangerous

$$\begin{bmatrix} 10^{-16} & 1 \\ 1 & 1 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} = \begin{bmatrix} 1 \\ 2 \end{bmatrix}$$


Multiplier: $l_{21} = \frac{1}{10^{-16}} = 10^{16}$


Row 2 Updates:

$$
a_{22}^{(2)} = 1 - \ell_{21}(1) = 1 - 10^{16} \;\longrightarrow\; -10^{16}
\qquad (\text{rounded --- the } {+}1 \text{ is completely swamped!})
$$

$$
b_{2}^{(2)} = 2 - \ell_{21}(1) = 2 - 10^{16} \;\longrightarrow\; -9,999,999,999,999,998
$$

Backward Substitution:

$$x_2 = \frac{-9,999,999,999,999,998}{-10^{16}} = 0.9999999999999998$$

$$10^{-16} x_1 + 0.9999999999999998 = 1.0$$

When the computer calculates $1.0 - 0.9999999999999998$, the result is exactly standard machine epsilon ($\approx 2.22 \times 10^{-16}$):

$$10^{-16} x_1 = 2.22 \times 10^{-16} \implies \mathbf{x_1 = 2.22}$$


Note:

* $\vert{}l\vert{} \gg 1$ (Large Multiplier): Multiplying the pivot row by $l$ inflates any floating-point roundoff errors in that row by $l$. Subtracting this huge vector from Row $j$ completely swamps the original, smaller values in Row $j$.

* $\vert{}l\vert{} \le 1$ (Small Multiplier): We are subtracting at most $100\%$ of the pivot row. Error expansion is strictly bounded, and original row data is preserved.

In [ ]:
eps = 1e-16

l_21 = 1/eps
print(l_21)

1e+16


In [ ]:
a22 = 1 - l_21
print(a22)

-1e+16


In [ ]:
b2 = 2 - l_21
print(b2)

-9999999999999998.0


In [ ]:
x2 = b2/a22
print(x2)

0.9999999999999998


In [ ]:
1 - x2

2.220446049250313e-16

In [ ]:
x1 = (1 - x2)/eps
print(x1)

2.220446049250313


### Partial Pivoting

For any $n \times n$ system, during the $k$-th step of forward elimination, we search the $k$-th column from row $k$ down to row $n$ for the element with the **largest absolute value**. If this maximum element is in row $p$ (where $p \ge k$), we interchange the $k$-th equation with the $p$-th equation. This strategy guarantees that our multipliers are never greater than 1 in magnitude, preventing catastrophic rounding errors. This process is referred to as **partial pivoting**.

**How to track row changes**

Here is the entire transformation condensed into a single equation chain, exactly as Strang formats it in his book:

$$
\begin{aligned}
& \begin{bmatrix} 1 & 2 & a & \mathbf{1} \\ 2 & 4 & b & \mathbf{2} \\ 3 & 7 & c & \mathbf{3} \end{bmatrix} \\[1.5ex]
\xrightarrow{\text{Swap } R_1 \leftrightarrow R_3}\quad
& \begin{bmatrix} 3 & 7 & c & \mathbf{3} \\ 2 & 4 & b & \mathbf{2} \\ 1 & 2 & a & \mathbf{1} \end{bmatrix} \\[1.5ex]
\xrightarrow{\text{Elim. Col 1}}\quad
& \begin{bmatrix} 3 & 7 & c & \mathbf{3} \\ 0 & -\tfrac{2}{3} & b - \tfrac{2}{3}c & \mathbf{2} \\ 0 & -\tfrac{1}{3} & a - \tfrac{1}{3}c & \mathbf{1} \end{bmatrix} \\[1.5ex]
\xrightarrow{\text{Elim. Col 2}}\quad
& \begin{bmatrix} 3 & 7 & c & \mathbf{3} \\ 0 & -\tfrac{2}{3} & b - \tfrac{2}{3}c & \mathbf{2} \\ 0 & 0 & a - \tfrac{1}{2}b & \mathbf{1} \end{bmatrix}
\end{aligned}
$$

Reading the bold tracking column of the final matrix gives $[3, 2, 1]^{T}$, which directly defines the permutation matrix $P$:

$$
P = \begin{bmatrix} 0 & 0 & 1 \\ 0 & 1 & 0 \\ 1 & 0 & 0 \end{bmatrix}
$$

In [ ]:
def doolittle_pa_lu(A):
    A = A.astype(float).copy(); n = A.shape[0]
    L = np.eye(n); U = np.zeros((n, n))
    P = np.eye(n)  # <--- NEW: Initialize Permutation Matrix

    for i in range(n):
        # ==================== NEW: PIVOT SELECTION ====================
        # 1. Compute unscaled pivot candidates for column i (from row i down)
        candidates = [A[p, i] - np.dot(L[p, :i], U[:i, i]) for p in range(i, n)]

        # 2. Find row index with the largest absolute pivot value
        pivot_row = i + np.argmax(np.abs(candidates))

        # 3. Swap rows in A, P, and previously calculated lower entries in L
        if pivot_row != i:
            A[[i, pivot_row]] = A[[pivot_row, i]]
            P[[i, pivot_row]] = P[[pivot_row, i]]
            L[[i, pivot_row], :i] = L[[pivot_row, i], :i]
        # ==============================================================

        for j in range(i, n):
            U[i, j] = A[i, j] - np.dot(L[i, :i], U[:i, j])

        if U[i, i] == 0:
            raise ValueError("Zero pivot encountered")

        for j in range(i + 1, n):
            L[j, i] = (A[j, i] - np.dot(L[j, :i], U[:i, i])) / U[i, i]

    return P, L, U  # <--- NEW: Returns P alongside L and U

In [ ]:
def solve_pa_lu(P, L, U, b):

    # Apply permutation vector/matrix to b
    Pb = P @ b

    # Solve L y = Pb using YOUR custom forward substitution
    y = forward_substitution(L, Pb)

    # Solve U x = y using YOUR custom back substitution
    x = back_substitution(U, y)

    return x

### Example

In [ ]:
import numpy as np

# 1. Define epsilon near float64 machine precision (1e-16)
eps = 1e-16

A = np.array([
    [eps, 1.0],
    [1.0, 1.0]
], dtype=float)

b = np.array([1.0, 2.0], dtype=float)

# 2. Solve using partial pivoted Doolittle LU
P, L, U = doolittle_pa_lu(A)
x_computed = solve_pa_lu(P, L, U, b)

# 3. True exact solution: x1 = 1/(1-eps) ≈ 1.0, x2 = (1-2eps)/(1-eps) ≈ 1.0
x_exact = np.array([1.0 / (1.0 - eps), (1.0 - 2.0 * eps) / (1.0 - eps)])

# --- Display Results ---
print("--- Matrix P (Permutation) ---")
print(P)

print("\n--- Matrix L ---")
print(L)

print("\n--- Matrix U ---")
print(U)

print("\n--- Numerical Solution Comparison ---")
print("Doolittle (With Pivoting) x   :", x_computed)
print("Exact Mathematical Solution x :", x_exact)
print("Relative Error in x1          :", abs(x_computed[0] - x_exact[0]) / x_exact[0] * 100, "%")

--- Matrix P (Permutation) ---
[[0. 1.]
 [1. 0.]]

--- Matrix L ---
[[1. 0.]
 [0. 1.]]

--- Matrix U ---
[[1. 1.]
 [0. 1.]]

--- Numerical Solution Comparison ---
Doolittle (With Pivoting) x   : [1. 1.]
Exact Mathematical Solution x : [1. 1.]
Relative Error in x1          : 2.2204460492503124e-14 %


---

**Connection to Production Solvers**

When you call np.linalg.solve, the underlying LAPACK routines (GESV and GETRF) execute the exact same partial pivoting algorithm you just built. To optimize memory, LAPACK overwrites the original matrix and tracks row swaps using a compact 1D integer array — conceptually identical to just adding an extra tracking column instead of storing a full $n \times n$ permutation matrix.

---

In [ ]:
import numpy as np
import scipy as sp

# np.linalg.solve?
sp.linalg.lu?

Signature:
sp.linalg.lu(
    a,
    permute_l=False,
    overwrite_a=False,
    check_finite=True,
    p_indices=False,
)
Docstring:
Compute LU decomposition of a matrix with partial pivoting.

The decomposition satisfies::

    A = P @ L @ U

where ``P`` is a permutation matrix, ``L`` lower triangular with unit
diagonal elements, and ``U`` upper triangular. If `permute_l` is set to
``True`` then ``L`` is returned already permuted and hence satisfying
``A = L @ U``.

Parameters
----------
a : (M, N) array_like
    Array to decompose
permute_l : bool, optional
    Perform the multiplication P*L (Default: do not permute)
overwrite_a : bool, optional
    Whether to overwrite data in a (may improve performance)
check_finite : bool, optional
    Whether to check that the input matrix contains only finite numbers.
    Disabling may give a performance gain, but may result in problems
    (crashes, non-termination) if the inputs do contain infinities or NaNs.
p_indices : bool, optional
    If 

### <span style="color:#c62828">4b. Scaled Partial Pivoting </span>

#### Can partial pivoting still be fooled? Yes.
Partial pivoting picks the largest **raw** number in the column. But if one equation is written in different **units** (say, one row measured in metres and another in micrometres), its coefficients can be huge. **Scaled partial pivoting (SPP)** removes this artificial scaling by judging each candidate pivot **relative to the largest entry in its own row**.

### The mathematical framework

**1. Scale factors.** For each row $i$, the largest magnitude coefficient:
$$s_i=\max_{1\le j\le n}|a_{ij}|.$$


**32. Pivot rule.** At step $k$, choose the pivot row $p$ (over rows $l\ge k$) that maximizes the **scaled ratio**:
$$\frac{|a_{pk}|}{s_p}\;\ge\;\frac{|a_{lk}|}{s_l}\quad\text{for all } l\ge k.$$


### Example (4-digit rounding by hand)

$$\begin{aligned}0.7\,x_1 + 1725\,x_2 &= 1739\\ 0.4352\,x_1 - 5.433\,x_2 &= 3.271\end{aligned}
\qquad\text{exact solution } (x_1,x_2)=(20,\,1).$$

**Scale factors:** $s_1=\max(0.7,1725)=1725$, $s_2=\max(0.4352,5.433)=5.433$.

**Scaled ratios in column 1:**
$$\frac{|a_{11}|}{s_1}=\frac{0.7}{1725}\approx 0.0004,\qquad
\frac{|a_{21}|}{s_2}=\frac{0.4352}{5.433}\approx 0.0801.$$

Since $0.0801>0.0004$, **swap rows** and pivot on $0.4352$. Without the swap, the small pivot $0.7$ produces a large multiplier and 4-digit rounding wrecks the answer (the naive result is $x_1\approx 17.14$ instead of $20$). SPP recovers $(20,1)$ accurately.


### Pseudocode — Gaussian elimination with SPP

```text
# Phase 1: scale factors
for i = 1 to n:
    s(i) = max_j |a(i,j)|
    if s(i) == 0: stop "no unique solution"

# Phase 2: forward elimination with scaled pivot choice
for k = 1 to n-1:
    choose p in {k..n} maximizing |a(l,k)| / s(l)
    if a(p,k) == 0: stop "no unique solution"
    swap rows k <-> p in a, in b, and in s
    for i = k+1 to n:
        m = a(i,k) / a(k,k)
        a(i, k:) = a(i, k:) - m * a(k, k:)
        b(i)     = b(i)     - m * b(k)

# Phase 3: back substitution (as before)
```


### Python — vectorized SPP solver

In [ ]:
def gauss_spp(A, b):
    "Solve A x = b by Gaussian elimination with scaled partial pivoting."
    A = A.astype(float).copy(); b = b.astype(float).copy(); n = len(b)
    s = np.max(np.abs(A), axis=1)                     # row scale factors
    if np.any(s == 0):
        raise ValueError("Row of zeros: no unique solution.")
    for k in range(n - 1):
        ratios = np.abs(A[k:, k]) / s[k:]             # scaled ratios
        p = np.argmax(ratios) + k
        if A[p, k] == 0:
            raise ValueError("No unique solution.")
        if p != k:
            A[[k, p]] = A[[p, k]]                      # swap rows...
            b[[k, p]] = b[[p, k]]                      # ...in b...
            s[[k, p]] = s[[p, k]]                      # ...and scale factors
        f = A[k+1:, k] / A[k, k]
        A[k+1:, k:] -= np.outer(f, A[k, k:])
        b[k+1:]     -= f * b[k]
    return back_substitution(A, b)

In [ ]:
# Example: The mixed-units system: SPP recovers (20, 1) in full double precision

A = np.array([[0.7,    1725.0],
              [0.4352,   -5.433]])
b = np.array([1739.0, 3.271])
print("SPP solution :", gauss_spp(A, b), " (exact = [20, 1])")
# print("residual     :", residual(A, gauss_spp(A, b), b))

# Also works on the zero-pivot 3x3 from Section 3
A3 = np.array([[6, -2,   3  ],
               [1, -1/3, 1/3],
               [1,  3,  -1  ]], dtype=float)
b3 = np.array([5, 2, 5], dtype=float)
# print("\nSPP on 3x3   :", gauss_spp(A3, b3), " residual", residual(A3, gauss_spp(A3, b3), b3))

SPP solution : [20.  1.]  (exact = [20, 1])


### Arithmetic cost
SPP adds only **comparisons** to choose the pivot — an extra $\mathcal O(n^2)$ of work on top of the $\mathcal O(n^3)$ elimination.

However, production libraries rarely use scaled partial pivoting because the extra calculations disrupt highly optimized memory caches. Instead, they pre-scale the entire matrix once beforehand (called equilibration) and then rely on standard partial pivoting.

## <span style="color:#00695c">5. Special Systems — Cholesky Decomposition</span>

Symmetric positive-definite systems — common in engineering*

### Symmetric Matrices

A symmetric matrix is a square matrix with $A = A^T$: the entries mirror across the main diagonal, so $a_{ij} = a_{ji}$.

For example:

$$A = \begin{bmatrix} 2 & 3 & 1 \\ 3 & 5 & 4 \\ 1 & 4 & 6 \end{bmatrix}, \qquad a_{ij}=a_{ji}.$$

Because $a_{ij} = a_{ji}$, we only need to store and compute half of the matrix. This means direct factorization of symmetric matrices is twice as fast and uses half the memory of standard $LU$ decomposition.

**Symmetric Positive Definite (SPD) Matrices**

A symmetric $n \times n$ matrix $A$ is positive definite if the scalar $\mathbf{x}^T A \mathbf{x} > 0$ for every nonzero vector $\mathbf{x}$.

For example, $A = \begin{bmatrix} 2 & -1 \\ -1 & 2 \end{bmatrix}$ is SPD:

$$\mathbf{x}^T A \mathbf{x} = 2x_1^2 - 2x_1x_2 + 2x_2^2 = x_1^2 + x_2^2 + (x_1-x_2)^2 > 0 \quad \text{for } \mathbf{x}\neq\mathbf{0}.$$

If a matrix is Symmetric Positive Definite (SPD), the largest elements are naturally anchored on the main diagonal. No partial pivoting is needed at all!

The algorithm is unconditionally stable against catastrophic roundoff errors.

For physical systems (like structural analysis or heat transfer), symmetric matrices are usually Positive Definite.

$$ A = LL^T = \begin{bmatrix} 2 & -1 \\ -1 & 2 \end{bmatrix} =
\begin{bmatrix} \sqrt{2} & 0 \\ -\frac{1}{\sqrt{2}} & \sqrt{\frac{3}{2}} \end{bmatrix}
\begin{bmatrix} \sqrt{2} & -\frac{1}{\sqrt{2}} \\ 0 & \sqrt{\frac{3}{2}} \end{bmatrix}$$

$$
A = LDL^T = \begin{bmatrix} 2 & -1 \\ -1 & 2 \end{bmatrix}
= \begin{bmatrix} 1 & 0 \\ -\tfrac{1}{2} & 1 \end{bmatrix}
\begin{bmatrix} 2 & 0 \\ 0 & \tfrac{3}{2} \end{bmatrix}
\begin{bmatrix} 1 & -\tfrac{1}{2} \\ 0 & 1 \end{bmatrix}
$$

**Cholesky Factorization: $A = LL^T$**

If $A$ is SPD, we do not need separate $L$ and $U$ matrices. Instead, we can factor $A$ into a lower triangular matrix $L$ multiplied by its own transpose $L^T$:

$$A = L L^T$$

This is called the Cholesky Factorization.

$$\begin{bmatrix} a_{11} & a_{21} & a_{31} \\ a_{21} & a_{22} & a_{32} \\ a_{31} & a_{32} & a_{33} \end{bmatrix} = \begin{bmatrix} l_{11} & 0 & 0 \\ l_{21} & l_{22} & 0 \\ l_{31} & l_{32} & l_{33} \end{bmatrix} \begin{bmatrix} l_{11} & l_{21} & l_{31} \\ 0 & l_{22} & l_{32} \\ 0 & 0 & l_{33} \end{bmatrix}$$

We calculate $L$ column by column (or row by row). Notice how every step only relies on previously computed $L$ values:

**Column 1**

* $l_{11} = \sqrt{a_{11}}$
* $l_{21} = \frac{a_{21}}{l_{11}}$
* $l_{31} = \frac{a_{31}}{l_{11}}$

**Column 2**

* $l_{22} = \sqrt{a_{22} - l_{21}^2}$
* $l_{32} = \frac{1}{l_{22}} \left( a_{32} - l_{31}l_{21} \right)$

**Column 3**

$l_{33} = \sqrt{a_{33} - (l_{31}^2 + l_{32}^2)}$


The General Formulas:

For diagonal entries ($j = i$):

$$l_{jj} = \sqrt{a_{jj} - \sum_{k=0}^{j-1} l_{jk}^2}$$

For off-diagonal entries ($i > j$):

$$l_{ij} = \frac{1}{l_{jj}} \left( a_{ij} - \sum_{k=0}^{j-1} l_{ik} l_{jk} \right)$$


## Python Implementation

In [ ]:
def cholesky_decomposition(A):
    """
    Computes the Cholesky factorization A = L L^T for a symmetric positive-definite matrix.
    """
    n = A.shape[0]
    L = np.zeros_like(A, dtype=float)

    for i in range(n):
        for j in range(i + 1):
            # Compute the dot product of the previously computed row entries
            sum_k = np.dot(L[i, :j], L[j, :j])

            if i == j:
                # Diagonal elements
                val = A[i, i] - sum_k
                if val <= 0:
                    raise ValueError("Matrix is not positive definite!")
                L[i, i] = np.sqrt(val)
            else:
                # Off-diagonal elements
                L[i, j] = (A[i, j] - sum_k) / L[j, j]

    return L

In [ ]:
# A simple 3x3 Symmetric Positive Definite matrix
A_sym = np.array([[4.0, 12.0, -16.0],
                  [12.0, 37.0, -43.0],
                  [-16.0, -43.0, 98.0]])

# 1. Compute L
L = cholesky_decomposition(A_sym)

# 2. Verify A = L * L.T
print("Computed L:\n", L)
print("\nL @ L.T:\n", L @ L.T)
print("\nDoes L @ L.T match A?", np.allclose(L @ L.T, A_sym))

Computed L:
 [[ 2.  0.  0.]
 [ 6.  1.  0.]
 [-8.  5.  3.]]

L @ L.T:
 [[  4.  12. -16.]
 [ 12.  37. -43.]
 [-16. -43.  98.]]

Does L @ L.T match A? True


### Arithmetic Operations: Cost and Memory

| Method | Operations (FLOPs) | Memory |
| :-- | :--: | :-- |
| Standard $LU$ factorization | $\approx \tfrac{2}{3} n^3$ | full $n \times n$ matrix |
| Cholesky factorization ($LL^T$) | $\approx \tfrac{1}{3} n^3$ | lower triangle only |

Cholesky uses half the operations and half the memory of $LU$.

### Example

$$
\begin{bmatrix} 4 & 12 & -16 \\ 12 & 37 & -43 \\ -16 & -43 & 98 \end{bmatrix}
\begin{bmatrix} x_1 \\ x_2 \\ x_3 \end{bmatrix}
= \begin{bmatrix} 1 \\ 2 \\ 3 \end{bmatrix}
$$

In [ ]:
import scipy.linalg as la

# 1. Define a simple 3x3 Symmetric Positive Definite matrix and load vector
A = np.array([
    [ 4.0,  12.0, -16.0],
    [12.0,  37.0, -43.0],
    [-16.0, -43.0,  98.0]
])
b = np.array([1.0, 2.0, 3.0])

# ==========================================
# Custom
# ==========================================

# Step A: Factorize A = L L^T
L = cholesky_decomposition(A)

# Step B: Solve L y = b
y = forward_substitution(L, b)

# Step C: Solve L^T x = y
# (Notice we use L.T because the upper matrix is the transpose of L)
x_custom = back_substitution(L.T, y)

# ==========================================
# SCIPY
# ==========================================

# cho_factor returns the factorized matrix and a boolean flag indicating if it's lower/upper
cho_matrix, lower_flag = la.cho_factor(A, lower=True)

# cho_solve handles both the forward and backward substitution internally
x_scipy = la.cho_solve((cho_matrix, lower_flag), b)

# ==========================================
# 4. VERIFICATION
# ==========================================
print("Custom Pipeline x:", x_custom)
print("SciPy Pipeline x :", x_scipy)
print("Perfect Match?   :", np.allclose(x_custom, x_scipy))

Custom Pipeline x: [28.583333 -7.666667  1.333333]
SciPy Pipeline x : [28.583333 -7.666667  1.333333]
Perfect Match?   : True


### What if the Matrix is Symmetric but NOT Positive Definite? ($LDL^T$)


$$A = LL^T = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}$$

$LL^T$ factorization gives imaginary elements

$$A = LL^T = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix}  = \begin{bmatrix} 1 & 0 \\ 2 & \sqrt{3}i \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 0 & \sqrt{3}i \end{bmatrix}$$

$LU$ factorization

$$A = LU = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ 2 & 1 \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 0 & -3 \end{bmatrix}$$

$LDL^T$ factorization

$$A = LDL^T = \begin{bmatrix} 1 & 2 \\ 2 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 0 \\ 2 & 1 \end{bmatrix} \begin{bmatrix} 1 & 0 \\ 0 & -3 \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 0 & 1 \end{bmatrix}$$

**Solving $Ax = b$ with $LDL^T$** takes three quick steps (since $LDL^T x = b$):

* Forward substitution: solve $Lz = b$ for $z$.
* Diagonal scaling: solve $Dy = z$, i.e. $y_i = z_i / d_{ii}$.
* Back substitution: solve $L^T x = y$ for $x$.

**Takeaway:** $LDL^T$ keeps the $\tfrac{1}{3}n^3$ speed and half-memory of Cholesky, but also handles indefinite matrices. LAPACK relies on it (with special pivoting) for exactly this reason.

To understand where the formulas come from, let's write out $A = LDL^T$ manually.

Because we are pulling the diagonal values into $D$, we force the main diagonal of $L$ to be entirely $1$s.

$$\begin{bmatrix} a_{11} & a_{21} & a_{31} \\ a_{21} & a_{22} & a_{32} \\ a_{31} & a_{32} & a_{33} \end{bmatrix} =  \underbrace{\begin{bmatrix} 1 & 0 & 0 \\ l_{21} & 1 & 0 \\ l_{31} & l_{32} & 1 \end{bmatrix}}_{L} \underbrace{\begin{bmatrix} d_1 & 0 & 0 \\ 0 & d_2 & 0 \\ 0 & 0 & d_3 \end{bmatrix}}_{D} \underbrace{\begin{bmatrix} 1 & l_{21} & l_{31} \\ 0 & 1 & l_{32} \\ 0 & 0 & 1 \end{bmatrix}}_{L^T}$$

**Step 1: Column 1**

* Top-left ($a_{11}$): $1 \cdot d_1 \cdot 1 = d_1 \implies \mathbf{d_1 = a_{11}}$
* Middle-left ($a_{21}$): $l_{21} \cdot d_1 \cdot 1 = a_{21} \implies \mathbf{l_{21} = \frac{a_{21}}{d_1}}$
* Bottom-left ($a_{31}$): $l_{31} \cdot d_1 \cdot 1 = a_{31} \implies \mathbf{l_{31} = \frac{a_{31}}{d_1}}$

**Step 2: Column 2**

* Middle-center ($a_{22}$): $l_{21} \cdot d_1 \cdot l_{21} + 1 \cdot d_2 \cdot 1 = a_{22} \implies \mathbf{d_2 = a_{22} - l_{21}^2 d_1}$
* Bottom-center ($a_{32}$): $l_{31} \cdot d_1 \cdot l_{21} + l_{32} \cdot d_2 \cdot 1 = a_{32} \implies \mathbf{l_{32} = \frac{1}{d_2} (a_{32} - l_{31} l_{21} d_1)}$

**Step 3: Column 3**

* Bottom-right ($a_{33}$): $l_{31} \cdot d_1 \cdot l_{31} + l_{32} \cdot d_2 \cdot l_{32} + 1 \cdot d_3 \cdot 1 = a_{33} \implies \mathbf{d_3 = a_{33} - (l_{31}^2 d_1 + l_{32}^2 d_2)}$

For the diagonal ($D$):

$$D_{jj} = A_{jj} - \sum_{k=0}^{j-1} L_{jk}^2 D_{kk}$$

For the off-diagonal ($L$):

$$L_{ij} = \frac{1}{D_{jj}} \left( A_{ij} - \sum_{k=0}^{j-1} L_{ik} L_{jk} D_{kk} \right)$$

In [ ]:
def ldlt_decomposition(A):
    """
    Computes the LDL^T factorization for a symmetric matrix.
    Safely handles indefinite matrices (negative eigenvalues).
    """
    n = A.shape[0]
    L = np.eye(n)       # L starts as an identity matrix (1s on the diagonal)
    D_vec = np.zeros(n) # Store D as a 1D vector first for easy math

    for j in range(n):
        # 1. Compute the diagonal element D[j, j]
        sum_D = np.sum((L[j, :j] ** 2) * D_vec[:j])
        D_vec[j] = A[j, j] - sum_D

        # 2. Compute the L elements below the diagonal in column j
        for i in range(j + 1, n):
            sum_L = np.sum(L[i, :j] * L[j, :j] * D_vec[:j])
            L[i, j] = (A[i, j] - sum_L) / D_vec[j]

    # Return L, and convert D_vec back into a 2D diagonal matrix for easy viewing
    return L, np.diag(D_vec)

In [ ]:
# Example: LDL^T on a symmetric INDEFINITE matrix (Cholesky would fail here)
A_ind = np.array([[1.0, 2.0],
                  [2.0, 1.0]])

L, D = ldlt_decomposition(A_ind)
print("L =\n", L)
print("D =\n", D)
print("\nReconstruct L D L^T =\n", L @ D @ L.T)
print("Matches A?", np.allclose(L @ D @ L.T, A_ind))

L =
 [[1. 0.]
 [2. 1.]]
D =
 [[ 1.  0.]
 [ 0. -3.]]

Reconstruct L D L^T =
 [[1. 2.]
 [2. 1.]]
Matches A? True


The Cholesky and $LDL^T$ factors are related by:

$$L_{\text{chol}} = L_{\text{ldl}} \sqrt{D}$$

(where $\sqrt{D}$ takes the square root of each diagonal entry of $D$).

If $LDL^T$ avoids square roots and handles both SPD and indefinite matrices, why is Cholesky still the favorite? Two practical reasons:

* $A = LL^T$ carries a single matrix $L$. In large simulations, one matrix is easier to store and pass than $L$ plus a separate $D$.
* The square root scales values down. For matrices with a huge value spread (multi-scale material simulations), Cholesky keeps intermediates smaller and guards against floating-point overflow.

**Engineering consensus:**

* System known to be positive definite → use **Cholesky**: the optimized gold standard.
* System that may buckle or go unstable → use **$LDL^T$**.

## <span style="color:#283593">Summary — Choosing a Direct Method</span>

| Method | Applies to | Cost (factor) | Reuse per new $\mathbf b$ | Stability | Key idea |
| :-- | :-- | :--: | :--: | :-- | :-- |
| **Gaussian elimination** | general square $A$ | $\tfrac23 n^3$ | — (redo) | needs pivoting | eliminate → back-substitute |
| **Gauss–Jordan / inverse** | general square $A$ | $\tfrac12 n^3$ | $n^2$ ($A^{-1}\mathbf b$) | needs pivoting | $[A\mid I]\to[I\mid A^{-1}]$ |
| **$A=LU$** | general square $A$ | $\tfrac23 n^3$ | $2n^2$ | needs pivoting | store multipliers in $L$ |
| **$PA=LU$** | general square $A$ | $\tfrac23 n^3$ | $2n^2$ | **stable** (partial pivot) | swap to largest $\lvert$pivot$\rvert$ |
| **Scaled partial pivoting** | general, mixed units | $\tfrac23 n^3$ | $2n^2$ | **stable** across scales | pivot by scaled ratio |
| **$LDL^{T}$** | symmetric | $\tfrac13 n^3$ | $2n^2$ | good (symmetric) | half storage, no square roots |
| **Cholesky $LL^{T}$** | symmetric **positive definite** | $\tfrac13 n^3$ | $2n^2$ | **excellent, no pivoting** | $\sqrt{\text{pivots}}$ into $L$ |

*You never need to compute $A^{-1}$ to solve $A\mathbf x=\mathbf b$; factor-and-substitute is faster and more accurate.*